In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv('../data/dataset_clean_max.csv')
df.head()

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,29460,131231,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,0.39006,0.10963,0.55097,-0.56104,1,NaN,0,0,7.9,NaN
1,67854,30597,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,0.03265,-0.40256,0.36218,0.86583,1,NaN,0,0,5.5,NaN
2,39470,45514,27,0,20326.87,720.0,25,73.61,492.57,4.6,...,-0.15637,0.57818,0.28902,-2.19864,1,NaN,0,0,7.2,NaN
3,24758,132414,45,49,38452.47,703.0,17,47.53,204.18,25.3,...,-1.02145,0.63908,-0.89190,-0.81592,1,NaN,0,0,4.4,NaN
4,73212,148536,37,46,NaN,594.0,13,99.95,734.09,12.8,...,-0.65771,0.08020,0.17606,0.86739,1,NaN,0,0,4.9,NaN


In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100
cols_to_drop = missing_percent[missing_percent > 85].index
df_train_clean = df.drop(columns=cols_to_drop)

# Garder que numériques + supprimer lignes NaN restantes
df_train_clean = df_train_clean.dropna()
df_train_clean = df_train_clean.select_dtypes(include=['int64', 'float64', 'bool'])
df_train_clean = df_train_clean.loc[:, df_train_clean.nunique() > 1]


X_train = df_train_clean.drop('target_is_fraud', axis=1)
y_train = df_train_clean['target_is_fraud']

model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Cross-validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')

# Entraînement
model.fit(X_train, y_train)

# Prédictions
y_train_pred = model.predict(X_train)


print("=== RÉSULTATS TRAIN ===")
print(f"CV Recall:  {cv_scores.mean():.4f}")
print(f"Accuracy:   {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Precision:  {precision_score(y_train, y_train_pred):.4f}")
print(f"Recall:     {recall_score(y_train, y_train_pred):.4f}")
print(f"F1-Score:   {f1_score(y_train, y_train_pred):.4f}")
print("\nMatrice de confusion:")
print(confusion_matrix(y_train, y_train_pred))


df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v3.csv')

df_test_clean = df_test.drop(columns=cols_to_drop, errors='ignore')
df_test_clean = df_test_clean.dropna()
df_test_clean = df_test_clean.select_dtypes(include=['int64', 'float64', 'bool'])

df_test_clean = df_test_clean[X_train.columns]

if 'target_is_fraud' in df_test_clean.columns:
    X_test = df_test_clean.drop('target_is_fraud', axis=1)
    y_test = df_test_clean['target_is_fraud']
else:
    X_test = df_test_clean

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

if 'target_is_fraud' in df_test_clean.columns:
    print(f"Accuracy:   {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision:  {precision_score(y_test, y_test_pred):.4f}")
    print(f"Recall:     {recall_score(y_test, y_test_pred):.4f}")
    print(f"F1-Score:   {f1_score(y_test, y_test_pred):.4f}")
    print("\nMatrice de confusion:")
    print(confusion_matrix(y_test, y_test_pred))


results_df = pd.DataFrame({
    'customer_id': df_test.index,
    'target': y_test_pred,
    'probability_0': y_test_proba[:, 0],
    'probability_1': y_test_proba[:, 1]
})
results_df.to_csv('../data/DecisionTree_predictions.csv', index=False)
print(results_df.head())

=== RÉSULTATS TRAIN ===
CV Recall:  0.9576
Accuracy:   0.9995
Precision:  0.9949
Recall:     0.9894
F1-Score:   0.9922

Matrice de confusion:
[[114142     18]
 [    38   3545]]


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- terms_accepted_flag
Feature names seen at fit time, yet now missing:
- account_id
- browser
- channel
- city
- country
- ...
